# Using AutoGen for Autonomous Internal Document QA and Policy Compliance

**Objective:** Demonstrate an intelligent multi-agent system that automatically checks company documents for compliance issues and routes them to the right people to fix them — an automated document assistant that never sleeps.

**Tools:** Python (Google Colab), AutoGen (AG2), OpenAI

**Prerequisite:** an OpenAI API key

### Steps
1. Install the required libraries
2. Import the libraries and load the API key
3. Create a configuration list and an LLM configuration
4. Create sample documents
5. Create specialized agents
6. Create a function to run the agents
7. Create a function to set up custom compliance rules
8. Define a function to set up webhook monitoring
9. Execute with try / except

> **How to run:** add your OpenAI key to Colab **Secrets** as `OPENAI_API_KEY`, then choose **Runtime → Run all**.

## Step 1: Install the required libraries

In [1]:
!pip install -q autogen "ag2[openai]" openai

## Step 2: Import the libraries and load the API key

Don't paste the key into the notebook. Store it in Colab's **Secrets** panel (🔑 icon in the left sidebar) as `OPENAI_API_KEY` and turn on *Notebook access*. If no secret is found, the cell asks for the key in a hidden input box.

In [2]:
import autogen
from typing import Dict, List, Optional
import json
import re
from datetime import datetime
import os
from getpass import getpass
from IPython.display import Markdown, display

# Load the API key from Colab Secrets, falling back to a hidden prompt
try:
    from google.colab import userdata
    api_key = userdata.get("OPENAI_API_KEY")
except Exception:
    api_key = None

if not api_key:
    api_key = getpass("Enter your OPENAI_API_KEY: ")

os.environ["OPENAI_API_KEY"] = api_key
print("API key loaded ✅" if os.environ.get("OPENAI_API_KEY") else "API key missing ❌")

API key loaded ✅


## Step 3: Create a configuration list and an LLM configuration

In [10]:
# Creating a variable for configuration
config_list = [
    {
        "model": "gpt-4o-mini",
        "api_key": os.environ.get("OPENAI_API_KEY"),
    }
]

# Create LLM configuration
llm_config = {
    "config_list": config_list,
    "temperature": 0,
}

## Step 4: Create sample documents

Three documents with deliberate compliance problems: a superseded-guideline reference, missing sections, and a missing signature / review date.

In [11]:
# Sample documents for demonstration
SAMPLE_DOCUMENTS = [
    {
        "id": "DOC001",
        "title": "Employee Remote Work Policy",
        "type": "HR Policy",
        "content": """
        Remote Work Policy
        Effective Date: January 2023

        1. Eligibility: All full-time employees
        2. Equipment: Company will provide laptop
        3. Work Hours: Standard 9-5 EST

        Note: This policy supersedes the 2021 remote work guidelines.
        """,
        "last_updated": "2023-01-15"
    },
    {
        "id": "DOC002",
        "title": "Data Security Manual",
        "type": "Technical Manual",
        "content": """
        Data Security Manual
        Version: 2.1

        1. Password Requirements: Minimum 8 characters
        2. Encryption: Use AES-256 for sensitive data

        Missing sections:
        - Incident Response Procedures
        - Data Retention Policy
        """,
        "last_updated": "2024-03-20"
    },
    {
        "id": "DOC003",
        "title": "Customer Service Guidelines",
        "type": "Operational Guidelines",
        "content": """
        Customer Service Guidelines

        1. Response Time: Within 24 hours
        2. Escalation: After 48 hours to supervisor
        3. Communication: Use template responses from 2019 handbook

        Approved by: [Missing Signature]
        Review Date: [Not Specified]
        """,
        "last_updated": "2024-11-10"
    }
]

print(f"Loaded {len(SAMPLE_DOCUMENTS)} sample documents")

Loaded 3 sample documents


## Step 5: Create specialized agents

| Agent | Role |
|---|---|
| **DocumentScanner** | Retrieves documents, extracts metadata, categorises them |
| **ComplianceChecker** | Finds issues and rates them Critical / Major / Minor |
| **ReviewCoordinator** | Routes issues by severity and creates action items with deadlines |
| **UserProxy** | Represents the human; kicks off the chat |

All four sit in a `GroupChat`, orchestrated by a `GroupChatManager`. The chat ends automatically when the ReviewCoordinator posts `TERMINATE`, and the final summary is built from the agents' real output, not hard-coded text.

In [12]:
def is_done(msg):
    """True once an agent's message contains the stop word TERMINATE"""
    return "TERMINATE" in (msg.get("content") or "")


# Define specialized agents
class DocumentComplianceSystem:
    def __init__(self):
        # Document Scanner Agent - Monitors and retrieves documents
        self.document_scanner = autogen.AssistantAgent(
            name="DocumentScanner",
            llm_config=llm_config,
            system_message="""You are a Document Scanner agent responsible for:
            1. Monitoring document repositories for new or updated documents
            2. Extracting document metadata (title, type, last updated date)
            3. Preparing documents for compliance analysis
            4. Identifying document categories (HR, Technical, Operational)

            Format your findings clearly with document ID, type, and key metadata.

            Stay in your role: only extract metadata and categorize documents.
            Do NOT identify compliance issues, assign severities, recommend actions,
            assign teams, or set due dates - other agents handle those."""
        )

        # Compliance Checker Agent - Analyzes documents for compliance issues
        self.compliance_checker = autogen.AssistantAgent(
            name="ComplianceChecker",
            llm_config=llm_config,
            system_message="""You are a Compliance Checker agent responsible for:
            1. Analyzing documents for compliance issues such as:
               - Missing required sections (approvals, dates, signatures)
               - Outdated references or superseded content
               - Inconsistent terminology or formatting
               - Version control issues
            2. Categorizing issues by severity (Critical, Major, Minor)
            3. Providing specific recommendations for remediation

            Always structure your analysis with: Issue Type | Severity | Details | Recommendation

            Stay in your role: do NOT route documents, assign teams, or set due dates -
            the ReviewCoordinator handles those."""
        )

        # Review Coordinator Agent - Routes documents and manages escalations
        self.review_coordinator = autogen.AssistantAgent(
            name="ReviewCoordinator",
            llm_config=llm_config,
            system_message="""You are a Review Coordinator agent responsible for:
            1. Processing compliance check results
            2. Determining routing based on severity:
               - Critical: Immediate escalation to Legal/Executive review
               - Major: Route to department head for approval
               - Minor: Send revision request to document owner
            3. Generating action items with deadlines
            4. Creating audit trail for compliance tracking

            Present your routing decisions as ONE markdown table with EXACTLY these
            6 columns, in this order, and one row per document:
            | Document | Issues | Severity | Routing Decision | Action Items | Due Date |
            In the Document column, write the document ID (e.g. DOC001), not its title.

            Set each Due Date from today's date using the document's highest severity:
               - Critical: 3 days
               - Major: 14 days
               - Minor: 30 days
            Write dates as YYYY-MM-DD.

            After the table, add a short "Audit Trail" bullet list.
            End your final message with the word TERMINATE on its own line."""
        )

        # User Proxy Agent - Represents human interaction points
        self.user_proxy = autogen.UserProxyAgent(
            name="UserProxy",
            human_input_mode="NEVER",
            max_consecutive_auto_reply=2,
            code_execution_config=False,
            is_termination_msg=is_done,
        )

        # Create group chat for multi-agent collaboration
        self.group_chat = autogen.GroupChat(
            agents=[
                self.document_scanner,
                self.compliance_checker,
                self.review_coordinator,
                self.user_proxy
            ],
            messages=[],
            max_round=10
        )

        self.manager = autogen.GroupChatManager(
            groupchat=self.group_chat,
            llm_config=llm_config,
            is_termination_msg=is_done,
        )

    def process_document_batch(self, documents: List[Dict]):
        """Process a batch of documents through the compliance workflow"""
        print("🚀 Starting AutoGen Document Compliance Workflow\n")
        print("=" * 60)

        # Convert documents to a formatted string for processing
        doc_summary = "\n\n".join([
            f"Document ID: {doc['id']}\n"
            f"Title: {doc['title']}\n"
            f"Type: {doc['type']}\n"
            f"Last Updated: {doc['last_updated']}\n"
            f"Content:\n{doc['content']}"
            for doc in documents
        ])

        # Initiate the multi-agent workflow
        today = datetime.now().strftime('%Y-%m-%d')
        initial_message = f"""
        New documents have been detected in the repository. Please process these documents through our compliance workflow:

        {doc_summary}

        Today's date is {today}. Set due dates relative to today.

        Workflow Steps:
        1. DocumentScanner: Analyze and categorize the documents
        2. ComplianceChecker: Perform compliance analysis on each document
        3. ReviewCoordinator: Determine routing and create action items

        Begin the analysis.
        """

        self.user_proxy.initiate_chat(
            self.manager,
            message=initial_message
        )

        return self.generate_compliance_report(documents)

    def _last_message_from(self, agent_name: str) -> Optional[str]:
        """Return the most recent message an agent posted in the group chat"""
        for msg in reversed(self.group_chat.messages):
            if msg.get("name") == agent_name and msg.get("content"):
                return msg["content"]
        return None

    @staticmethod
    def count_severities(routing_table: str) -> Dict[str, int]:
        """Count documents per severity, finding the Severity column from the table header"""
        counts = {"Critical": 0, "Major": 0, "Minor": 0}
        severity_col = None
        for line in routing_table.splitlines():
            line = line.strip()
            if not line.startswith("|"):
                continue
            cells = [c.strip().strip("*").strip() for c in line.strip("|").split("|")]
            if severity_col is None:
                # The first table row is the header: locate the Severity column
                lowered = [c.lower() for c in cells]
                if "severity" in lowered:
                    severity_col = lowered.index("severity")
                continue
            if all(set(c) <= set("-: ") for c in cells):
                continue  # the |---|---| separator row
            if severity_col < len(cells):
                for level in counts:
                    if level.lower() in cells[severity_col].lower():
                        counts[level] += 1
                        break
        return counts

    def generate_compliance_report(self, documents: List[Dict]):
        """Build the summary from what the agents actually found in this run"""
        print("\n" + "=" * 60)
        print("📊 COMPLIANCE WORKFLOW SUMMARY")
        print("=" * 60)
        print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"Documents Processed: {len(documents)}")
        print(f"Chat turns: {len(self.group_chat.messages)}")

        routing = self._last_message_from("ReviewCoordinator")
        if not routing:
            print("\n⚠️ The ReviewCoordinator did not post a routing decision this run.")
            return None

        routing = routing.replace("TERMINATE", "").strip()
        counts = self.count_severities(routing)

        print("\nDocuments by severity:")
        for level, icon in [("Critical", "🔴"), ("Major", "🟠"), ("Minor", "🟡")]:
            print(f"  {icon} {level}: {counts[level]}")

        print("\nRouting decisions from the ReviewCoordinator:\n")
        display(Markdown(routing))
        print("\nWorkflow Demonstration Complete ✅")
        return routing

## Step 6: Create a function to run the agents

In [13]:
# Demonstration function
def run_demo():
    """Run the AutoGen document compliance demonstration"""
    print("🤖 AutoGen Document Compliance System Demo")
    print("=" * 60)
    print("Demonstrating key AutoGen features:")
    print("✓ Multi-agent collaboration")
    print("✓ Declarative agent configuration")
    print("✓ Group chat orchestration")
    print("✓ Automated workflow execution")
    print("=" * 60 + "\n")

    # Initialize the compliance system
    compliance_system = DocumentComplianceSystem()

    # Process the sample documents
    compliance_system.process_document_batch(SAMPLE_DOCUMENTS)

## Step 7: Create a function to set up custom compliance rules

In [14]:
# Additional helper functions for extended functionality
def create_custom_compliance_rules():
    """Define custom compliance rules for different document types"""
    return {
        "HR Policy": {
            "required_sections": ["Effective Date", "Eligibility", "Approval"],
            "max_age_months": 12,
            "requires_signature": True
        },
        "Technical Manual": {
            "required_sections": ["Version", "Security Requirements", "Incident Response"],
            "max_age_months": 6,
            "requires_signature": False
        },
        "Operational Guidelines": {
            "required_sections": ["Procedures", "Escalation", "Review Date"],
            "max_age_months": 12,
            "requires_signature": True
        }
    }

print(json.dumps(create_custom_compliance_rules(), indent=2))

{
  "HR Policy": {
    "required_sections": [
      "Effective Date",
      "Eligibility",
      "Approval"
    ],
    "max_age_months": 12,
    "requires_signature": true
  },
  "Technical Manual": {
    "required_sections": [
      "Version",
      "Security Requirements",
      "Incident Response"
    ],
    "max_age_months": 6,
    "requires_signature": false
  },
  "Operational Guidelines": {
    "required_sections": [
      "Procedures",
      "Escalation",
      "Review Date"
    ],
    "max_age_months": 12,
    "requires_signature": true
  }
}


## Step 8: Define a function to set up webhook monitoring

In [15]:
def setup_monitoring_webhook():
    """Setup webhook for continuous document monitoring"""
    webhook_config = {
        "endpoint": "/api/document-updates",
        "events": ["document.created", "document.updated"],
        "frequency": "real-time"
    }
    return webhook_config

print(json.dumps(setup_monitoring_webhook(), indent=2))

{
  "endpoint": "/api/document-updates",
  "events": [
    "document.created",
    "document.updated"
  ],
  "frequency": "real-time"
}


## Step 9: Include the try / except statements and execute

Runs the full workflow. No input is needed — the chat stops by itself and prints the summary.

In [16]:
if __name__ == "__main__":
    try:
        run_demo()
    except Exception as e:
        print(f"\n❌ Error: {e}")
        print("Check that OPENAI_API_KEY is set in Colab Secrets and Step 1 ran successfully.")

🤖 AutoGen Document Compliance System Demo
Demonstrating key AutoGen features:
✓ Multi-agent collaboration
✓ Declarative agent configuration
✓ Group chat orchestration
✓ Automated workflow execution

🚀 Starting AutoGen Document Compliance Workflow

UserProxy (to chat_manager):


        New documents have been detected in the repository. Please process these documents through our compliance workflow:

        Document ID: DOC001
Title: Employee Remote Work Policy
Type: HR Policy
Last Updated: 2023-01-15
Content:

        Remote Work Policy
        Effective Date: January 2023

        1. Eligibility: All full-time employees
        2. Equipment: Company will provide laptop
        3. Work Hours: Standard 9-5 EST

        Note: This policy supersedes the 2021 remote work guidelines.
        

Document ID: DOC002
Title: Data Security Manual
Type: Technical Manual
Last Updated: 2024-03-20
Content:

        Data Security Manual
        Version: 2.1

        1. Password Requirements: Minimum

| Document | Issues | Severity | Routing Decision | Action Items | Due Date |
|----------|--------|----------|------------------|---------------|-----------|
| DOC001   | Missing Required Sections, Outdated References | Major | Route to department head for approval | Include an approval section with signatures and specify a review date; Clarify changes from 2021 guidelines | 2026-11-19 |
| DOC002   | Missing Required Sections | Critical | Immediate escalation to Legal/Executive review | Add the missing sections: Incident Response Procedures and Data Retention Policy | 2026-11-08 |
| DOC003   | Missing Required Sections, Inconsistent Terminology | Major | Route to department head for approval | Include necessary signatures for approval and specify a review date; Update reference to the most current handbook | 2026-11-19 |

- Audit Trail:
  - Document DOC001 identified with major and minor issues.
  - Document DOC002 identified with critical issues.
  - Document DOC003 identified with major and minor issues.


Workflow Demonstration Complete ✅


## Conclusion

I've built an AutoGen-powered multi-agent system that checks company documents for compliance issues, identifies gaps, and routes them to the right people for timely resolution.

**Ideas to extend it:** feed `create_custom_compliance_rules()` into the ComplianceChecker's prompt so each document is checked against its type's required sections and maximum age, or load real documents from a folder instead of `SAMPLE_DOCUMENTS`.